# LayerNorm Kernel — All-in-One Test & Benchmark Notebook

Tests both variants (`ForgeLayerNormLiger`, `ForgeLayerNormUnsloth`) end-to-end.

**Run order:** *Cell → Run All*. Each section prints its own pass/fail line and appends to a global `RESULTS` dict; the final cell prints a roll-up table.

**Sections:**
1. Setup (imports, device check, helpers)
2. Forward correctness vs `F.layer_norm`
3. Backward correctness (Liger full, Unsloth dX-only)
4. `torch.autograd.gradcheck` at fp64
5. Edge cases (eps sweep, non-power-of-2 H, constant input, state_dict)
6. Variant comparison (Liger ↔ Unsloth, in-place `dY` overwrite)
7. Latency benchmarks
8. Peak VRAM benchmarks
9. Memory bandwidth (% of A100 peak)
10. CUDA kernel launch count
11. Alignment impact diagnostic
12. Final summary table


## 1 — Setup

In [1]:
import gc, math, os, sys, time
from pathlib import Path

import torch
import torch.nn.functional as F

# Make `from kernels.layernorm import ...` work regardless of where the notebook
# is launched from. This file lives at kernels/layernorm/, so the repo root is 2 up.
_HERE = Path.cwd()
for candidate in (_HERE, _HERE.parent, _HERE.parent.parent):
    if (candidate / 'kernels' / 'layernorm' / 'layernorm_kernel.py').exists():
        sys.path.insert(0, str(candidate))
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError('Could not find kernels/layernorm/ from cwd: ' + str(_HERE))
print('repo root:', REPO_ROOT)

from kernels.layernorm import (
    ForgeLayerNormLiger,
    ForgeLayerNormLigerFunction,
    ForgeLayerNormUnsloth,
    ForgeLayerNormUnslothFunction,
)

assert torch.cuda.is_available(), 'CUDA is required'
DEVICE = 'cuda'
GPU_NAME = torch.cuda.get_device_name(0)
SM_COUNT = torch.cuda.get_device_properties(0).multi_processor_count
print(f'GPU: {GPU_NAME}  ({SM_COUNT} SMs)')
if 'A100' not in GPU_NAME:
    print('WARNING: this suite is calibrated for A100; numbers will differ on other GPUs')

torch.manual_seed(3407)
torch.cuda.manual_seed_all(3407)

# Roll-up results — each section appends here. Last cell prints the summary.
RESULTS = []
def record(section, name, status, detail=''):
    RESULTS.append((section, name, status, detail))
    icon = {'PASS': '✓', 'FAIL': '✗', 'INFO': '·'}.get(status, '?')
    print(f'  {icon} {name}  {detail}')

repo root: /workspace/kernel-POCs


GPU: NVIDIA A100-SXM4-80GB  (108 SMs)


In [2]:
# Shared helpers (matches tests/_helpers.py but self-contained)

A100_PEAK_BW = 1555e9  # bytes/sec

SHAPES_SMALL  = [(2, 8, 1024), (1, 1, 128), (4, 512, 4096)]
SHAPES_DESIGN = [(4, 2048, 4096), (8, 2048, 4096)]
SHAPES_SWEEP  = SHAPES_SMALL + SHAPES_DESIGN + [(8, 4096, 4096)]

TOL = {
    torch.bfloat16: dict(rtol=1e-2, atol=1e-2),
    torch.float16:  dict(rtol=1e-3, atol=1e-3),
    torch.float32:  dict(rtol=1e-5, atol=1e-5),
    torch.float64:  dict(rtol=1e-7, atol=1e-7),
}

def make_inputs(shape, dtype, requires_grad=True, nontrivial_affine=True):
    B, S, H = shape
    X = torch.randn(B, S, H, dtype=dtype, device=DEVICE, requires_grad=requires_grad)
    if nontrivial_affine:
        W = (torch.randn(H, dtype=dtype, device=DEVICE) * 0.5 + 1.0).detach().requires_grad_(requires_grad)
        Bp = (torch.randn(H, dtype=dtype, device=DEVICE) * 0.1).detach().requires_grad_(requires_grad)
    else:
        W  = torch.ones(H,  dtype=dtype, device=DEVICE, requires_grad=requires_grad)
        Bp = torch.zeros(H, dtype=dtype, device=DEVICE, requires_grad=requires_grad)
    dY = torch.randn(B, S, H, dtype=dtype, device=DEVICE)
    return X, W, Bp, dY

def clone_leaf(t, requires_grad):
    return t.detach().clone().requires_grad_(requires_grad)

def elem_size(dtype):
    return 2 if dtype in (torch.bfloat16, torch.float16) else 4

def n_rows(shape):
    return shape[0] * shape[1]

def fwd_bytes(shape, dtype):
    e = elem_size(dtype); n = n_rows(shape); d = shape[-1]
    return (n*d*e + 2*d*e) + (n*d*e + 2*n*4)

def bwd_bytes_liger(shape, dtype):
    e = elem_size(dtype); n = n_rows(shape); d = shape[-1]
    progs = min(n, SM_COUNT)
    return (2*n*d*e + d*e + 2*n*4) + (n*d*e + 2*progs*d*4)

def bwd_bytes_unsloth(shape, dtype):
    e = elem_size(dtype); n = n_rows(shape); d = shape[-1]
    return (2*n*d*e + d*e + 2*n*4) + n*d*e

def sync_time_ms(fn, warmup=25, iters=100):
    for _ in range(warmup): fn()
    torch.cuda.synchronize()
    times = []
    for _ in range(iters):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        fn()
        torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    times.sort()
    return times[len(times) // 2]

def peak_mem_mb(fn):
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    fn(); torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated() / (1024**2)

print('helpers ready')

helpers ready


## 2 — Forward correctness vs `F.layer_norm`

Both variants, swept across shapes and dtypes.

In [3]:
print('--- Forward correctness ---')
VARIANTS = {
    'liger':   ForgeLayerNormLigerFunction,
    'unsloth': ForgeLayerNormUnslothFunction,
}
DTYPES = [torch.bfloat16, torch.float16, torch.float32]

for variant_name, fn in VARIANTS.items():
    for shape in SHAPES_SWEEP:
        for dtype in DTYPES:
            X, W, Bp, _ = make_inputs(shape, dtype, requires_grad=False)
            out = fn.apply(X, W, Bp, 1e-6)
            ref = F.layer_norm(X, (shape[-1],), W, Bp, 1e-6)
            try:
                torch.testing.assert_close(out, ref, **TOL[dtype])
                status = 'PASS'
                detail = ''
            except AssertionError as e:
                status = 'FAIL'
                detail = f'max_abs={(out - ref).abs().max().item():.3e}'
            record('forward', f'{variant_name} {shape} {dtype}', status, detail)


--- Forward correctness ---


  ✓ liger (2, 8, 1024) torch.bfloat16  
  ✓ liger (2, 8, 1024) torch.float16  
  ✓ liger (2, 8, 1024) torch.float32  
  ✓ liger (1, 1, 128) torch.bfloat16  
  ✓ liger (1, 1, 128) torch.float16  
  ✓ liger (1, 1, 128) torch.float32  
  ✓ liger (4, 512, 4096) torch.bfloat16  
  ✓ liger (4, 512, 4096) torch.float16  
  ✓ liger (4, 512, 4096) torch.float32  
  ✓ liger (4, 2048, 4096) torch.bfloat16  
  ✓ liger (4, 2048, 4096) torch.float16  
  ✓ liger (4, 2048, 4096) torch.float32  
  ✓ liger (8, 2048, 4096) torch.bfloat16  
  ✓ liger (8, 2048, 4096) torch.float16  
  ✓ liger (8, 2048, 4096) torch.float32  
  ✓ liger (8, 4096, 4096) torch.bfloat16  
  ✓ liger (8, 4096, 4096) torch.float16  
  ✓ liger (8, 4096, 4096) torch.float32  
  ✓ unsloth (2, 8, 1024) torch.bfloat16  
  ✓ unsloth (2, 8, 1024) torch.float16  
  ✓ unsloth (2, 8, 1024) torch.float32  
  ✓ unsloth (1, 1, 128) torch.bfloat16  
  ✓ unsloth (1, 1, 128) torch.float16  
  ✓ unsloth (1, 1, 128) torch.float32  
  ✓ unsloth (4, 5

## 3 — Backward correctness

- **Liger:** dX, dW, dB all checked against eager autograd.
- **Unsloth:** dX checked (W, B frozen). Then verify the `dW = dB = None` contract.

In [4]:
print('--- Liger backward (full dX, dW, dB) — fp64 comparison ---')
# Run BOTH the Forge kernel and the reference at fp64.
# This exercises the new fp64 accumulator path (Fix #2) end-to-end and
# eliminates fp32 reduction noise — the dW error should now be at fp64 ULP scale.
for shape in SHAPES_SWEEP:
    for dtype in (torch.bfloat16, torch.float64):
        X, W, Bp, dY = make_inputs(shape, dtype, requires_grad=False)

        # Forge kernel at native dtype (bf16 or fp64)
        Xf = clone_leaf(X, True); Wf = clone_leaf(W, True); Bf = clone_leaf(Bp, True)
        ForgeLayerNormLigerFunction.apply(Xf, Wf, Bf, 1e-6).backward(dY)

        # Reference in fp64; cast grads back to dtype for tolerance check.
        Xr = X.detach().double().requires_grad_(True)
        Wr = W.detach().double().requires_grad_(True)
        Br = Bp.detach().double().requires_grad_(True)
        F.layer_norm(Xr, (shape[-1],), Wr, Br, 1e-6).backward(dY.double())
        dX_ref = Xr.grad.to(dtype)
        dW_ref = Wr.grad.to(dtype)
        dB_ref = Br.grad.to(dtype)

        ok = True; reason = ''
        for name, a, b in [('dX', Xf.grad, dX_ref), ('dW', Wf.grad, dW_ref), ('dB', Bf.grad, dB_ref)]:
            try:
                torch.testing.assert_close(a, b, **TOL[dtype])
            except AssertionError:
                ok = False
                reason = f'{name} max_abs={(a - b).abs().max().item():.3e}'
                break
        record('backward-liger', f'{shape} {dtype}', 'PASS' if ok else 'FAIL', reason)


--- Liger backward (full dX, dW, dB) — fp64 comparison ---


  ✓ (2, 8, 1024) torch.bfloat16  


  ✓ (2, 8, 1024) torch.float64  
  ✓ (1, 1, 128) torch.bfloat16  


  ✓ (1, 1, 128) torch.float64  
  ✓ (4, 512, 4096) torch.bfloat16  


  ✓ (4, 512, 4096) torch.float64  
  ✓ (4, 2048, 4096) torch.bfloat16  


  ✓ (4, 2048, 4096) torch.float64  
  ✓ (8, 2048, 4096) torch.bfloat16  


  ✓ (8, 2048, 4096) torch.float64  
  ✓ (8, 4096, 4096) torch.bfloat16  


  ✓ (8, 4096, 4096) torch.float64  


In [5]:
print('--- Unsloth backward (dX-only, W/B frozen) ---')
for shape in SHAPES_SWEEP:
    for dtype in (torch.bfloat16, torch.float32):
        X, W, Bp, dY = make_inputs(shape, dtype, requires_grad=False)

        Xf = clone_leaf(X, True); Wf = clone_leaf(W, False); Bf = clone_leaf(Bp, False)
        ForgeLayerNormUnslothFunction.apply(Xf, Wf, Bf, 1e-6).backward(dY.clone())

        Xr = clone_leaf(X, True); Wr = clone_leaf(W, False); Br = clone_leaf(Bp, False)
        F.layer_norm(Xr, (shape[-1],), Wr, Br, 1e-6).backward(dY)

        try:
            torch.testing.assert_close(Xf.grad, Xr.grad, **TOL[dtype])
            status, detail = 'PASS', ''
        except AssertionError:
            status = 'FAIL'
            detail = f'dX max_abs={(Xf.grad - Xr.grad).abs().max().item():.3e}'
        record('backward-unsloth', f'{shape} {dtype}', status, detail)


--- Unsloth backward (dX-only, W/B frozen) ---
  ✓ (2, 8, 1024) torch.bfloat16  
  ✓ (2, 8, 1024) torch.float32  
  ✓ (1, 1, 128) torch.bfloat16  
  ✓ (1, 1, 128) torch.float32  
  ✓ (4, 512, 4096) torch.bfloat16  
  ✓ (4, 512, 4096) torch.float32  
  ✓ (4, 2048, 4096) torch.bfloat16  
  ✓ (4, 2048, 4096) torch.float32  
  ✓ (8, 2048, 4096) torch.bfloat16  
  ✓ (8, 2048, 4096) torch.float32  
  ✓ (8, 4096, 4096) torch.bfloat16  
  ✓ (8, 4096, 4096) torch.float32  


In [6]:
print('--- Unsloth dW/dB = None contract ---')
X, W, Bp, dY = make_inputs((2, 8, 1024), torch.float32, requires_grad=False)
Xf = clone_leaf(X, True); Wf = clone_leaf(W, True); Bf = clone_leaf(Bp, True)
ForgeLayerNormUnslothFunction.apply(Xf, Wf, Bf, 1e-6).backward(dY.clone())

ok = (Xf.grad is not None) and (Wf.grad is None) and (Bf.grad is None)
record('backward-unsloth', 'dW/dB None contract',
       'PASS' if ok else 'FAIL',
       f'X.grad={Xf.grad is not None}, W.grad={Wf.grad}, B.grad={Bf.grad}')

--- Unsloth dW/dB = None contract ---
  ✓ dW/dB None contract  X.grad=True, W.grad=None, B.grad=None


## 4 — `torch.autograd.gradcheck` at fp64

In [7]:
print('--- fp64 gradcheck ---')
GC_SHAPE = (2, 4, 8)
B, S, H = GC_SHAPE

# Liger — gradcheck against (X, W, B) all requiring grad
X = torch.randn(B, S, H, dtype=torch.float64, device=DEVICE, requires_grad=True)
W = (torch.randn(H, dtype=torch.float64, device=DEVICE) * 0.5 + 1.0).detach().requires_grad_(True)
Bp = (torch.randn(H, dtype=torch.float64, device=DEVICE) * 0.1).detach().requires_grad_(True)

try:
    ok = torch.autograd.gradcheck(ForgeLayerNormLigerFunction.apply, (X, W, Bp, 1e-6),
                                  eps=1e-6, atol=1e-5, rtol=1e-3, raise_exception=True)
    record('gradcheck', 'liger (X, W, B)', 'PASS' if ok else 'FAIL')
except Exception as e:
    record('gradcheck', 'liger (X, W, B)', 'FAIL', str(e)[:120])

# Unsloth — only X requires grad
X = torch.randn(B, S, H, dtype=torch.float64, device=DEVICE, requires_grad=True)
W = (torch.randn(H, dtype=torch.float64, device=DEVICE) * 0.5 + 1.0).detach()
Bp = (torch.randn(H, dtype=torch.float64, device=DEVICE) * 0.1).detach()

try:
    ok = torch.autograd.gradcheck(ForgeLayerNormUnslothFunction.apply, (X, W, Bp, 1e-6),
                                  eps=1e-6, atol=1e-5, rtol=1e-3, raise_exception=True)
    record('gradcheck', 'unsloth (X only)', 'PASS' if ok else 'FAIL')
except Exception as e:
    record('gradcheck', 'unsloth (X only)', 'FAIL', str(e)[:120])

# Document the failure when W, B require grad on Unsloth (contract)
X = torch.randn(B, S, H, dtype=torch.float64, device=DEVICE, requires_grad=True)
W = (torch.randn(H, dtype=torch.float64, device=DEVICE) * 0.5 + 1.0).detach().requires_grad_(True)
Bp = (torch.randn(H, dtype=torch.float64, device=DEVICE) * 0.1).detach().requires_grad_(True)
try:
    torch.autograd.gradcheck(ForgeLayerNormUnslothFunction.apply, (X, W, Bp, 1e-6),
                             eps=1e-6, atol=1e-5, rtol=1e-3, raise_exception=True)
    record('gradcheck', 'unsloth (X, W, B) [expected fail]', 'FAIL', 'unexpectedly passed')
except Exception:
    record('gradcheck', 'unsloth (X, W, B) [expected fail]', 'PASS', 'failed as designed')


--- fp64 gradcheck ---


  ✓ liger (X, W, B)  
  ✓ unsloth (X only)  


  ✓ unsloth (X, W, B) [expected fail]  failed as designed


## 5 — Edge cases

In [8]:
print('--- eps sweep ---')
for variant, fn in VARIANTS.items():
    for eps in [1e-7, 1e-6, 1e-5, 1e-4, 1e-3]:
        X, W, Bp, _ = make_inputs((2, 8, 1024), torch.bfloat16, requires_grad=False)
        out = fn.apply(X, W, Bp, eps)
        ref = F.layer_norm(X, (1024,), W, Bp, eps)
        try:
            torch.testing.assert_close(out, ref, **TOL[torch.bfloat16])
            record('edge-eps', f'{variant} eps={eps}', 'PASS')
        except AssertionError as e:
            record('edge-eps', f'{variant} eps={eps}', 'FAIL', str(e)[:60])


--- eps sweep ---
  ✓ liger eps=1e-07  
  ✓ liger eps=1e-06  
  ✓ liger eps=1e-05  
  ✓ liger eps=0.0001  
  ✓ liger eps=0.001  


  ✓ unsloth eps=1e-07  
  ✓ unsloth eps=1e-06  
  ✓ unsloth eps=1e-05  
  ✓ unsloth eps=0.0001  
  ✓ unsloth eps=0.001  


In [9]:
print('--- non-power-of-2 H ---')
for variant, fn in VARIANTS.items():
    for H in [3, 17, 4097, 8193]:
        X, W, Bp, _ = make_inputs((2, 4, H), torch.float32, requires_grad=False)
        out = fn.apply(X, W, Bp, 1e-6)
        ref = F.layer_norm(X, (H,), W, Bp, 1e-6)
        try:
            torch.testing.assert_close(out, ref, rtol=1e-5, atol=1e-5)
            record('edge-H', f'{variant} H={H}', 'PASS')
        except AssertionError as e:
            record('edge-H', f'{variant} H={H}', 'FAIL', str(e)[:60])


--- non-power-of-2 H ---
  ✓ liger H=3  
  ✓ liger H=17  
  ✓ liger H=4097  
  ✓ liger H=8193  
  ✓ unsloth H=3  
  ✓ unsloth H=17  
  ✓ unsloth H=4097  
  ✓ unsloth H=8193  


In [10]:
print('--- constant input (var=0, output must equal B) ---')
for variant, fn in VARIANTS.items():
    X = torch.ones(2, 8, 256, dtype=torch.float32, device=DEVICE)
    W = torch.ones(256, dtype=torch.float32, device=DEVICE)
    Bp = torch.zeros(256, dtype=torch.float32, device=DEVICE)
    out = fn.apply(X, W, Bp, 1e-6)
    ok = torch.isfinite(out).all().item() and torch.allclose(out, Bp.expand_as(out), atol=1e-5)
    record('edge-constant', variant, 'PASS' if ok else 'FAIL')


--- constant input (var=0, output must equal B) ---
  ✓ liger  
  ✓ unsloth  


In [11]:
print('--- state_dict round-trip ---')
for variant, Module in (('liger', ForgeLayerNormLiger), ('unsloth', ForgeLayerNormUnsloth)):
    m1 = Module(1024, dtype=torch.bfloat16, device=DEVICE)
    with torch.no_grad():
        m1.weight.copy_(torch.randn_like(m1.weight))
        m1.bias.copy_(torch.randn_like(m1.bias))
    m2 = Module(1024, dtype=torch.bfloat16, device=DEVICE)
    m2.load_state_dict(m1.state_dict())
    X = torch.randn(2, 8, 1024, dtype=torch.bfloat16, device=DEVICE)
    ok = torch.equal(m1(X), m2(X))
    record('edge-statedict', variant, 'PASS' if ok else 'FAIL')


--- state_dict round-trip ---
  ✓ liger  
  ✓ unsloth  


## 6 — Variant comparison (Liger ↔ Unsloth)

In [12]:
print('--- variant parity ---')
SHAPE = (4, 2048, 4096)
X, W, Bp, dY = make_inputs(SHAPE, torch.float32, requires_grad=False)

# Forward parity at fp32
out_l = ForgeLayerNormLigerFunction.apply(X, W, Bp, 1e-6)
out_u = ForgeLayerNormUnslothFunction.apply(X, W, Bp, 1e-6)
try:
    torch.testing.assert_close(out_l, out_u, rtol=1e-5, atol=1e-5)
    record('variant', 'forward Liger==Unsloth (fp32)', 'PASS')
except AssertionError as e:
    record('variant', 'forward Liger==Unsloth (fp32)', 'FAIL', str(e)[:60])

# dX parity with frozen W, B
Xl = clone_leaf(X, True); Wl = clone_leaf(W, False); Bl = clone_leaf(Bp, False)
ForgeLayerNormLigerFunction.apply(Xl, Wl, Bl, 1e-6).backward(dY)
Xu = clone_leaf(X, True); Wu = clone_leaf(W, False); Bu = clone_leaf(Bp, False)
ForgeLayerNormUnslothFunction.apply(Xu, Wu, Bu, 1e-6).backward(dY.clone())
try:
    torch.testing.assert_close(Xl.grad, Xu.grad, rtol=1e-5, atol=1e-5)
    record('variant', 'dX Liger==Unsloth (frozen W/B)', 'PASS')
except AssertionError as e:
    record('variant', 'dX Liger==Unsloth (frozen W/B)', 'FAIL', str(e)[:60])

# In-place dY overwrite (Unsloth)
X, W, Bp, _ = make_inputs(SHAPE, torch.float32, requires_grad=False)
Xf = clone_leaf(X, True); Wf = clone_leaf(W, False); Bf = clone_leaf(Bp, False)
out = ForgeLayerNormUnslothFunction.apply(Xf, Wf, Bf, 1e-6)
dY_2d = torch.randn(SHAPE[0]*SHAPE[1], SHAPE[2], dtype=torch.float32, device=DEVICE).contiguous()
dY_before = dY_2d.clone()
out.backward(dY_2d.view(*SHAPE))
overwritten = not torch.equal(dY_2d, dY_before)
matches = torch.equal(dY_2d.view(*SHAPE), Xf.grad)
record('variant', 'Unsloth bwd overwrites dY in-place',
       'PASS' if (overwritten and matches) else 'FAIL')


--- variant parity ---
  ✓ forward Liger==Unsloth (fp32)  
  ✓ dX Liger==Unsloth (frozen W/B)  
  ✓ Unsloth bwd overwrites dY in-place  


## 7 — Latency benchmarks (eager / `torch.compile` / Liger / Unsloth)

Times forward, and forward+backward, on the two design shapes.

In [13]:
print('--- latency ---')
for shape in SHAPES_DESIGN:
    for dtype in (torch.bfloat16, torch.float32):
        X, W, Bp, dY = make_inputs(shape, dtype, requires_grad=False)
        eps = 1e-6

        compiled = torch.compile(
            lambda x, w, b: F.layer_norm(x, (shape[-1],), w, b, eps),
            mode='reduce-overhead',
        )
        compiled(X, W, Bp)  # warm compile

        fwd = {
            'eager':   sync_time_ms(lambda: F.layer_norm(X, (shape[-1],), W, Bp, eps)),
            'compile': sync_time_ms(lambda: compiled(X, W, Bp)),
            'liger':   sync_time_ms(lambda: ForgeLayerNormLigerFunction.apply(X, W, Bp, eps)),
            'unsloth': sync_time_ms(lambda: ForgeLayerNormUnslothFunction.apply(X, W, Bp, eps)),
        }

        def eager_fb():
            Xg = clone_leaf(X, True); Wg = clone_leaf(W, True); Bg = clone_leaf(Bp, True)
            F.layer_norm(Xg, (shape[-1],), Wg, Bg, eps).backward(dY)
        def liger_fb():
            Xg = clone_leaf(X, True); Wg = clone_leaf(W, True); Bg = clone_leaf(Bp, True)
            ForgeLayerNormLigerFunction.apply(Xg, Wg, Bg, eps).backward(dY)
        def unsloth_fb():
            Xg = clone_leaf(X, True); Wg = clone_leaf(W, False); Bg = clone_leaf(Bp, False)
            ForgeLayerNormUnslothFunction.apply(Xg, Wg, Bg, eps).backward(dY.clone())

        fb = {
            'eager':   sync_time_ms(eager_fb, warmup=10, iters=30),
            'liger':   sync_time_ms(liger_fb, warmup=10, iters=30),
            'unsloth': sync_time_ms(unsloth_fb, warmup=10, iters=30),
        }

        print(f'\n  shape={shape}  dtype={dtype}')
        print(f"  {'impl':<10} {'fwd (ms)':>10} {'fwd+bwd (ms)':>14} {'fwd vs eager':>14}")
        for name in ('eager', 'compile', 'liger', 'unsloth'):
            f = fwd.get(name, float('nan'))
            fb_v = fb.get(name, float('nan'))
            sp = fwd['eager'] / f if f else float('nan')
            print(f'  {name:<10} {f:>10.4f} {fb_v:>14.4f} {sp:>13.2f}x')

        record('latency', f'{shape} {dtype}', 'INFO',
               f"liger fwd={fwd['liger']:.3f}ms ({fwd['eager']/fwd['liger']:.2f}x) | "
               f"unsloth fwd={fwd['unsloth']:.3f}ms ({fwd['eager']/fwd['unsloth']:.2f}x)")


--- latency ---



  shape=(4, 2048, 4096)  dtype=torch.bfloat16
  impl         fwd (ms)   fwd+bwd (ms)   fwd vs eager
  eager          0.1504         0.5517          1.00x
  compile        0.2456            nan          0.61x
  liger          0.1549         0.6601          0.97x
  unsloth        0.1544         0.4552          0.97x
  · (4, 2048, 4096) torch.bfloat16  liger fwd=0.155ms (0.97x) | unsloth fwd=0.154ms (0.97x)



  shape=(4, 2048, 4096)  dtype=torch.float32
  impl         fwd (ms)   fwd+bwd (ms)   fwd vs eager
  eager          0.2435         0.9974          1.00x
  compile        0.4018            nan          0.61x
  liger          0.2274         0.7978          1.07x
  unsloth        0.2267         0.7437          1.07x
  · (4, 2048, 4096) torch.float32  liger fwd=0.227ms (1.07x) | unsloth fwd=0.227ms (1.07x)



  shape=(8, 2048, 4096)  dtype=torch.bfloat16
  impl         fwd (ms)   fwd+bwd (ms)   fwd vs eager
  eager          0.2376         0.9849          1.00x
  compile        0.4277            nan          0.56x
  liger          0.2285         0.7304          1.04x
  unsloth        0.2281         0.7487          1.04x
  · (8, 2048, 4096) torch.bfloat16  liger fwd=0.228ms (1.04x) | unsloth fwd=0.228ms (1.04x)



  shape=(8, 2048, 4096)  dtype=torch.float32
  impl         fwd (ms)   fwd+bwd (ms)   fwd vs eager
  eager          0.4642         1.9329          1.00x
  compile        0.7470            nan          0.62x
  liger          0.3812         1.2336          1.22x
  unsloth        0.3801         1.4290          1.22x
  · (8, 2048, 4096) torch.float32  liger fwd=0.381ms (1.22x) | unsloth fwd=0.380ms (1.22x)


## 8 — Peak VRAM

In [14]:
print('--- peak VRAM ---')
for shape in SHAPES_DESIGN:
    dtype = torch.bfloat16
    X, W, Bp, _ = make_inputs(shape, dtype, requires_grad=False)
    eps = 1e-6

    # Allocate dY inside each function so it counts toward peak.
    # Liger reads dY_local and allocates a separate DX buffer.
    # Unsloth writes DX in place over dY_local (no separate allocation).
    def eager_fb():
        Xg = clone_leaf(X, True); Wg = clone_leaf(W, True); Bg = clone_leaf(Bp, True)
        dY_local = torch.randn(*shape, dtype=dtype, device=DEVICE)
        F.layer_norm(Xg, (shape[-1],), Wg, Bg, eps).backward(dY_local)
    def liger_fb():
        Xg = clone_leaf(X, True); Wg = clone_leaf(W, True); Bg = clone_leaf(Bp, True)
        dY_local = torch.randn(*shape, dtype=dtype, device=DEVICE)
        ForgeLayerNormLigerFunction.apply(Xg, Wg, Bg, eps).backward(dY_local)
    def unsloth_fb():
        Xg = clone_leaf(X, True); Wg = clone_leaf(W, False); Bg = clone_leaf(Bp, False)
        dY_local = torch.randn(*shape, dtype=dtype, device=DEVICE)
        # NO .clone() — Unsloth overwrites dY_local in place to hold dX
        ForgeLayerNormUnslothFunction.apply(Xg, Wg, Bg, eps).backward(dY_local)

    fwd_peak = {
        'eager':   peak_mem_mb(lambda: F.layer_norm(X, (shape[-1],), W, Bp, eps)),
        'liger':   peak_mem_mb(lambda: ForgeLayerNormLigerFunction.apply(X, W, Bp, eps)),
        'unsloth': peak_mem_mb(lambda: ForgeLayerNormUnslothFunction.apply(X, W, Bp, eps)),
    }
    fb_peak = {
        'eager':   peak_mem_mb(eager_fb),
        'liger':   peak_mem_mb(liger_fb),
        'unsloth': peak_mem_mb(unsloth_fb),
    }

    expected_saving = n_rows(shape) * shape[-1] * elem_size(dtype) / (1024**2)

    print(f'\n  shape={shape}  dtype={dtype}')
    print(f"  {'impl':<10} {'fwd (MB)':>10} {'fwd+bwd (MB)':>14} {'vs eager fwd+bwd':>18}")
    for name in ('eager', 'liger', 'unsloth'):
        print(f"  {name:<10} {fwd_peak[name]:>10.1f} {fb_peak[name]:>14.1f} {fb_peak['eager'] - fb_peak[name]:>+17.1f}")
    print(f'  expected Unsloth in-place savings vs Liger: {expected_saving:.1f} MB')
    print(f"  observed Liger - Unsloth fwd+bwd:           {fb_peak['liger'] - fb_peak['unsloth']:+.1f} MB")

    record('memory', f'{shape}', 'INFO',
           f"liger={fb_peak['liger']:.0f}MB  unsloth={fb_peak['unsloth']:.0f}MB  "
           f"saving={fb_peak['liger']-fb_peak['unsloth']:+.0f}MB")

--- peak VRAM ---



  shape=(4, 2048, 4096)  dtype=torch.bfloat16
  impl         fwd (MB)   fwd+bwd (MB)   vs eager fwd+bwd
  eager          3776.6         3968.6              +0.0
  liger          3776.6         3972.0              -3.4
  unsloth        3776.6         3904.6             +64.0
  expected Unsloth in-place savings vs Liger: 64.0 MB
  observed Liger - Unsloth fwd+bwd:           +67.4 MB
  · (4, 2048, 4096)  liger=3972MB  unsloth=3905MB  saving=+67MB



  shape=(8, 2048, 4096)  dtype=torch.bfloat16
  impl         fwd (MB)   fwd+bwd (MB)   vs eager fwd+bwd
  eager          3968.6         4352.6              +0.0
  liger          3968.6         4356.0              -3.4
  unsloth        3968.6         4224.6            +128.0
  expected Unsloth in-place savings vs Liger: 128.0 MB
  observed Liger - Unsloth fwd+bwd:           +131.4 MB
  · (8, 2048, 4096)  liger=4356MB  unsloth=4225MB  saving=+131MB


## 9 — Memory bandwidth (% of A100 peak)

Achieved GB/s = analytical bytes ÷ measured time. Reference: A100 40GB peak 1555 GB/s.

In [15]:
print('--- bandwidth ---')
for shape in SHAPES_DESIGN:
    dtype = torch.bfloat16
    X, W, Bp, dY = make_inputs(shape, dtype, requires_grad=False)
    eps = 1e-6

    fwd_l = sync_time_ms(lambda: ForgeLayerNormLigerFunction.apply(X, W, Bp, eps))
    fwd_u = sync_time_ms(lambda: ForgeLayerNormUnslothFunction.apply(X, W, Bp, eps))

    def liger_fb():
        Xg = clone_leaf(X, True); Wg = clone_leaf(W, True); Bg = clone_leaf(Bp, True)
        ForgeLayerNormLigerFunction.apply(Xg, Wg, Bg, eps).backward(dY)
    def unsloth_fb():
        Xg = clone_leaf(X, True); Wg = clone_leaf(W, False); Bg = clone_leaf(Bp, False)
        ForgeLayerNormUnslothFunction.apply(Xg, Wg, Bg, eps).backward(dY.clone())

    fb_l = sync_time_ms(liger_fb, warmup=10, iters=30)
    fb_u = sync_time_ms(unsloth_fb, warmup=10, iters=30)
    bwd_l = max(fb_l - fwd_l, 1e-3)
    bwd_u = max(fb_u - fwd_u, 1e-3)

    b_fwd  = fwd_bytes(shape, dtype)
    b_bw_l = bwd_bytes_liger(shape, dtype)
    b_bw_u = bwd_bytes_unsloth(shape, dtype)

    def fmt(label, t, b):
        gbps = b / (t/1000) / 1e9
        return f'  {label:<22} {t:>9.4f} ms   {gbps:>8.1f} GB/s   {gbps / (A100_PEAK_BW/1e9) * 100:>5.1f}%'

    print(f'\n  shape={shape}  dtype={dtype}  peak={A100_PEAK_BW/1e9:.0f} GB/s')
    print(fmt('liger fwd',         fwd_l, b_fwd))
    print(fmt('unsloth fwd',       fwd_u, b_fwd))
    print(fmt('liger bwd (est)',   bwd_l, b_bw_l))
    print(fmt('unsloth bwd (est)', bwd_u, b_bw_u))

    gbps_l = b_fwd / (fwd_l/1000) / 1e9
    gbps_u = b_fwd / (fwd_u/1000) / 1e9
    record('bandwidth', f'{shape}', 'INFO',
           f'liger fwd={gbps_l:.0f} GB/s ({gbps_l/(A100_PEAK_BW/1e9)*100:.0f}%) | '
           f'unsloth fwd={gbps_u:.0f} GB/s ({gbps_u/(A100_PEAK_BW/1e9)*100:.0f}%)')


--- bandwidth ---

  shape=(4, 2048, 4096)  dtype=torch.bfloat16  peak=1555 GB/s
  liger fwd                 0.1513 ms      887.9 GB/s    57.1%
  unsloth fwd               0.1534 ms      875.8 GB/s    56.3%
  liger bwd (est)           0.6279 ms      326.4 GB/s    21.0%
  unsloth bwd (est)         0.3068 ms      656.5 GB/s    42.2%
  · (4, 2048, 4096)  liger fwd=888 GB/s (57%) | unsloth fwd=876 GB/s (56%)



  shape=(8, 2048, 4096)  dtype=torch.bfloat16  peak=1555 GB/s
  liger fwd                 0.2266 ms     1185.4 GB/s    76.2%
  unsloth fwd               0.2261 ms     1187.7 GB/s    76.4%
  liger bwd (est)           0.5096 ms      797.3 GB/s    51.3%
  unsloth bwd (est)         0.5220 ms      771.6 GB/s    49.6%
  · (8, 2048, 4096)  liger fwd=1185 GB/s (76%) | unsloth fwd=1188 GB/s (76%)


## 10 — CUDA kernel launch count

PyTorch eager runs ~10+ small kernels for one LN. Triton should fuse to ~1 fwd + ~1 bwd.

In [16]:
print('--- CUDA kernel launches ---')
from torch.profiler import ProfilerActivity, profile

SHAPE = (4, 2048, 4096)
X, W, Bp, dY = make_inputs(SHAPE, torch.bfloat16, requires_grad=False)
eps = 1e-6

def count_kernels(fn):
    for _ in range(3): fn()
    torch.cuda.synchronize()
    with profile(activities=[ProfilerActivity.CUDA], record_shapes=False) as prof:
        fn()
    torch.cuda.synchronize()
    return sum(1 for evt in prof.events() if evt.device_type == torch.autograd.DeviceType.CUDA)

def eager_fb():
    Xg = clone_leaf(X, True); Wg = clone_leaf(W, True); Bg = clone_leaf(Bp, True)
    F.layer_norm(Xg, (SHAPE[-1],), Wg, Bg, eps).backward(dY)
def liger_fb():
    Xg = clone_leaf(X, True); Wg = clone_leaf(W, True); Bg = clone_leaf(Bp, True)
    ForgeLayerNormLigerFunction.apply(Xg, Wg, Bg, eps).backward(dY)
def unsloth_fb():
    Xg = clone_leaf(X, True); Wg = clone_leaf(W, False); Bg = clone_leaf(Bp, False)
    ForgeLayerNormUnslothFunction.apply(Xg, Wg, Bg, eps).backward(dY.clone())

counts = {
    'eager':   count_kernels(eager_fb),
    'liger':   count_kernels(liger_fb),
    'unsloth': count_kernels(unsloth_fb),
}
print(f'\n  shape={SHAPE}  dtype=bf16  (fwd+bwd)')
for k, v in counts.items():
    print(f'  {k:<10} {v} CUDA events')
record('launches', f'{SHAPE} fwd+bwd', 'INFO',
       f"eager={counts['eager']}  liger={counts['liger']}  unsloth={counts['unsloth']}")


--- CUDA kernel launches ---

  shape=(4, 2048, 4096)  dtype=bf16  (fwd+bwd)
  eager      6 CUDA events
  liger      11 CUDA events
  unsloth    6 CUDA events
  · (4, 2048, 4096) fwd+bwd  eager=6  liger=11  unsloth=6


## 11 — Alignment impact (power-of-2 vs not)

In [17]:
print('--- alignment impact (Liger fwd) ---')
n = 4 * 2048
rows = []
for H in (4096, 4097, 8192, 8193):
    X, W, Bp, _ = make_inputs((1, n, H), torch.bfloat16, requires_grad=False)
    t = sync_time_ms(lambda: ForgeLayerNormLigerFunction.apply(X, W, Bp, 1e-6))
    bs = 1
    while bs < H: bs *= 2
    waste = (1 - H / bs) * 100
    rows.append((H, bs, waste, t))

print(f"\n  rows={n}  dtype=bf16")
print(f"  {'hidden':>7} {'block':>7} {'masked':>8}    {'fwd (ms)':>10}")
for H, bs, waste, t in rows:
    print(f'  {H:>7} {bs:>7} {waste:>7.1f}%   {t:>10.4f}')

record('alignment', 'H sweep', 'INFO',
       'see table above — power-of-2 H should be fastest per row')


--- alignment impact (Liger fwd) ---



  rows=8192  dtype=bf16
   hidden   block   masked      fwd (ms)
     4096    4096     0.0%       0.1594
     4097    8192    50.0%       0.2296
     8192    8192     0.0%       0.2384
     8193   16384    50.0%       0.4100
  · H sweep  see table above — power-of-2 H should be fastest per row


## 12 — Final summary

In [18]:
print('=' * 80)
print('SUMMARY')
print('=' * 80)

by_status = {'PASS': 0, 'FAIL': 0, 'INFO': 0}
by_section = {}
fails = []
for section, name, status, detail in RESULTS:
    by_status[status] = by_status.get(status, 0) + 1
    by_section.setdefault(section, {'PASS': 0, 'FAIL': 0, 'INFO': 0})
    by_section[section][status] = by_section[section].get(status, 0) + 1
    if status == 'FAIL':
        fails.append((section, name, detail))

print(f"\nOverall: {by_status['PASS']} PASS, {by_status['FAIL']} FAIL, {by_status['INFO']} INFO")
print(f"\n{'Section':<22} {'PASS':>5} {'FAIL':>5} {'INFO':>5}")
print('-' * 42)
for section, counts in by_section.items():
    print(f"{section:<22} {counts['PASS']:>5} {counts['FAIL']:>5} {counts['INFO']:>5}")

if fails:
    print(f'\n{len(fails)} FAILURES:')
    for section, name, detail in fails:
        print(f'  ✗ [{section}] {name}  {detail}')
else:
    print('\nAll hard gates passed.')

print(f'\nGPU: {GPU_NAME}  ({SM_COUNT} SMs)')


SUMMARY

Overall: 89 PASS, 0 FAIL, 10 INFO

Section                 PASS  FAIL  INFO
------------------------------------------
forward                   36     0     0
backward-liger            12     0     0
backward-unsloth          13     0     0
gradcheck                  3     0     0
edge-eps                  10     0     0
edge-H                     8     0     0
edge-constant              2     0     0
edge-statedict             2     0     0
variant                    3     0     0
latency                    0     0     4
memory                     0     0     2
bandwidth                  0     0     2
launches                   0     0     1
alignment                  0     0     1

All hard gates passed.

GPU: NVIDIA A100-SXM4-80GB  (108 SMs)
